<a href="https://colab.research.google.com/github/xD0ri4nx/bot_investitiV2/blob/main/bot_invetitii_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit yfinance scikit-learn tensorflow pandas numpy google-genai datasets transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 76.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import pipeline
from google.colab import userdata

# ==============================================================================
# 1. SECURE CONFIGURATION & LOCAL NLP INITIALIZATION
# ==============================================================================
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("[CRITICAL ERROR] Secret 'HF_TOKEN' not found. Please add it to Colab Secrets.")
    raise

# NOTE: Adjust this list! If you only need Apple right now, change to: TICKERS = ["AAPL"]
TICKERS = ["ETH-USD"]

# Institutional Backtest Target: ~10 years of data
MAX_ARTICLES_PER_ASSET = 10000

device = 0 if torch.cuda.is_available() else -1
if device == 0:
    print("🚀 GPU Detected! FinBERT will run with hardware acceleration.")
else:
    print("⚠️ No GPU detected. FinBERT will run on CPU (slower).")

print("Loading FinBERT Financial NLP Model into local memory...")
sentiment_analyzer = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device)
print("FinBERT Model Loaded Successfully!")

# ==============================================================================
# 2. FINBERT REASONING ENGINE (UPGRADED WITH GPU BATCHING)
# ==============================================================================
def analyze_daily_news(headlines_list):
    """Scores headlines using GPU Batching (batch_size=16) for maximum speed."""
    if not headlines_list:
        return 0.0

    # Truncate to 512 chars to prevent model indexing errors on massive articles
    safe_headlines = [str(h)[:512] for h in headlines_list]

    daily_score = 0.0
    valid_articles = 0

    try:
        # THE BATCHING OPTIMIZATION: Process 16 articles simultaneously on the GPU
        results = sentiment_analyzer(safe_headlines, batch_size=16)

        for result in results:
            label = result['label']
            if label == 'positive':
                daily_score += 1.0
            elif label == 'negative':
                daily_score += -1.0
            else:
                daily_score += 0.0

            valid_articles += 1

    except Exception as e:
        # If a severely corrupted batch crashes the pipeline, skip it safely
        return 0.0

    if valid_articles > 0:
        return round(daily_score / valid_articles, 4)
    return 0.0

# ==============================================================================
# 3. FAULT-TOLERANT DATA STREAMING PIPELINE (WITH CIRCUIT BREAKER)
# ==============================================================================
def fetch_raw_historical_news(ticker):
    print(f"\nStreaming Massive Hugging Face dataset for {ticker}...")

    ASSET_NAMES = {
        "AAPL": "apple",
        "NVDA": "nvidia",
        "MSFT": "microsoft",
        "GOOGL": "google",
        "SPY": "s&p 500",
        "BTC-USD": "bitcoin",
        "ETH-USD": "ethereum"
    }

    search_name = ASSET_NAMES.get(ticker, ticker.lower())

    try:
        dataset = load_dataset(
            "Brianferrell787/financial-news-multisource",
            split="train",
            streaming=True,
            token=HF_TOKEN
        )
    except Exception as e:
        print(f"[!] HuggingFace connection error: {e}")
        return {}

    news_dict = {}
    extracted_count = 0
    rows_scanned = 0

    # NEW: THE CIRCUIT BREAKER
    # If the script reads 3 million rows, it forces a stop to prevent RAM crashes.
    MAX_SCAN_DEPTH = 3000000

    try:
        for row in dataset:
            rows_scanned += 1
            text = str(row.get('text', ''))
            extra = str(row.get('extra_fields', ''))

            if search_name in text.lower() or ticker in extra:
                raw_date = row.get('date')
                try:
                    clean_date = pd.to_datetime(raw_date).strftime('%Y-%m-%d')
                except:
                    continue

                if clean_date not in news_dict:
                    news_dict[clean_date] = []

                news_dict[clean_date].append(text)
                extracted_count += 1

                if extracted_count % 500 == 0:
                    print(f"   ...Found {extracted_count}/{MAX_ARTICLES_PER_ASSET} articles for {ticker}...")

                if extracted_count >= MAX_ARTICLES_PER_ASSET:
                    break

            # The Circuit Breaker Trigger
            if rows_scanned >= MAX_SCAN_DEPTH:
                print(f"\n   [!] WARNING: Scanned {MAX_SCAN_DEPTH} rows. Stopping search to protect Colab RAM.")
                break

    except Exception as e:
        print(f"\n   [!] HUGGING FACE SERVER CRASH INTERCEPTED: {e}")

    print(f"   -> Saving the {extracted_count} articles we successfully grabbed.")
    print(f"[SUCCESS] Extracted {extracted_count} articles across {len(news_dict)} trading days for {ticker}.")
    return news_dict

# ==============================================================================
# 4. MEMORY-SAFE BATCH EXECUTION
# ==============================================================================
def run_sentiment_miner():
    print("\n--- STARTING OPTIMIZED NLP PIPELINE ---")

    for ticker in TICKERS:
        print(f"\n================ Processing {ticker} ================")
        historical_news = fetch_raw_historical_news(ticker)

        results = []
        total_days = len(historical_news)

        if total_days == 0:
            print(f"⚠️ [WARNING] No data found for {ticker}, skipping.")
            continue

        print(f"[{ticker}] Scoring {total_days} days using GPU Batching... Please wait.")

        for date, headlines in historical_news.items():
            score = analyze_daily_news(headlines)
            results.append({"Date": date, "Sentiment_Score": score})

        df_results = pd.DataFrame(results)
        df_results['Date'] = pd.to_datetime(df_results['Date'])
        df_results.sort_values('Date', inplace=True)

        output_filename = f"{ticker}_sentiment.csv"
        df_results.to_csv(output_filename, index=False)
        print(f"✅ [SAVED] {output_filename} is ready! (Total days scored: {total_days})")

        # ----------------------------------------------------------------------
        # GARBAGE COLLECTION TO PREVENT RAM CRASHES
        # ----------------------------------------------------------------------
        print(f"🧹 Sweeping memory logs to protect Colab RAM...")
        del historical_news
        del results
        del df_results
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # ----------------------------------------------------------------------

run_sentiment_miner()

🚀 GPU Detected! FinBERT will run with hardware acceleration.
Loading FinBERT Financial NLP Model into local memory...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT Model Loaded Successfully!

--- STARTING OPTIMIZED NLP PIPELINE ---

================ Processing ETH-USD ================

Streaming Massive Hugging Face dataset for ETH-USD...


Resolving data files:   0%|          | 0/131 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/131 [00:00<?, ?it/s]

   ...Found 500/10000 articles for ETH-USD...
   ...Found 1000/10000 articles for ETH-USD...
   ...Found 1500/10000 articles for ETH-USD...
   ...Found 2000/10000 articles for ETH-USD...

   [!] WARNING: Scanned 3000000 rows. Stopping search to protect Colab RAM.
   -> Saving the 2139 articles we successfully grabbed.
[SUCCESS] Extracted 2139 articles across 813 trading days for ETH-USD.
[ETH-USD] Scoring 813 days using GPU Batching... Please wait.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ [SAVED] ETH-USD_sentiment.csv is ready! (Total days scored: 813)
🧹 Sweeping memory logs to protect Colab RAM...


In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Bidirectional, Conv1D, GRU, LSTM,
    BatchNormalization, GaussianNoise, LayerNormalization, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
from tensorflow.keras.regularizers import l2
import tensorflow as tf
import warnings
import os
import gc

# ---- 1. SYSTEM CONFIGURATION & T4 GPU OPTIMIZATION ----
tf.random.set_seed(42)

# Enable Mixed Precision (FP16) - Unleashes NVIDIA T4 Tensor Cores
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

st.set_page_config(page_title="Institutional Quant AI", layout="wide", initial_sidebar_state="expanded")

# ---- CUSTOM CSS (GLASSMORPHISM) ----
st.markdown("""
<style>
    .stApp { background: linear-gradient(135deg, #0b132b, #1c2541, #3a506b); color: #ffffff; }
    header[data-testid="stHeader"] { background: transparent !important; }
    [data-testid="stSidebar"] {
        background: rgba(28, 37, 65, 0.4) !important;
        backdrop-filter: blur(15px);
        -webkit-backdrop-filter: blur(15px);
        border-right: 1px solid rgba(255, 255, 255, 0.1);
    }
    [data-testid="stMetric"] {
        background: rgba(255, 255, 255, 0.05); backdrop-filter: blur(12px);
        -webkit-backdrop-filter: blur(12px); border: 1px solid rgba(255, 255, 255, 0.1);
        border-radius: 12px; padding: 20px; box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.3);
        transition: transform 0.2s ease, box-shadow 0.2s ease;
    }
    [data-testid="stMetric"]:hover {
        transform: translateY(-2px); box-shadow: 0 12px 40px 0 rgba(0, 0, 0, 0.4);
        background: rgba(255, 255, 255, 0.08);
    }
    div.stButton > button {
        background: linear-gradient(90deg, #3a506b, #1c2541); color: white;
        border: 1px solid rgba(255, 255, 255, 0.2); border-radius: 8px;
        padding: 10px 24px; font-weight: 600; letter-spacing: 0.5px;
        transition: all 0.3s ease; box-shadow: 0 4px 15px rgba(0,0,0,0.2); width: 100%;
    }
    div.stButton > button:hover {
        background: linear-gradient(90deg, #5bc0be, #3a506b);
        border: 1px solid rgba(255, 255, 255, 0.5);
        box-shadow: 0 0 20px rgba(91, 192, 190, 0.4); transform: translateY(-1px);
    }
    [data-testid="stStatusWidget"] {
        background: rgba(0, 0, 0, 0.2) !important; backdrop-filter: blur(10px);
        border: 1px solid rgba(255, 255, 255, 0.1); border-radius: 10px;
    }
    .streamlit-expanderHeader { background: rgba(255, 255, 255, 0.05); border-radius: 8px; }
    h1, h2, h3, p, span, div, label { color: #e0e0e0; }
    [data-testid="stMetricValue"] { color: #ffffff !important; }

    *:focus, *:focus-visible, *:focus-within { outline: none !important; }
    .stSelectbox [data-baseweb="select"], .stDateInput [data-baseweb="input"] { background-color: transparent !important; }
    div[data-baseweb="select"] > div, div[data-baseweb="input"] > div {
        background: linear-gradient(90deg, #1c2541, #3a506b) !important;
        border: 1px solid rgba(255, 255, 255, 0.2) !important; border-radius: 8px !important;
        box-shadow: none !important; transition: all 0.3s ease !important;
    }
    div[data-baseweb="select"] > div:focus-within, div[data-baseweb="input"] > div:focus-within {
        border-color: #5bc0be !important; box-shadow: 0 0 0 1px #5bc0be !important;
    }
    div[data-baseweb="select"] > div:hover, div[data-baseweb="input"] > div:hover {
        background: linear-gradient(90deg, #2a3a5a, #4a6a8a) !important;
        border: 1px solid rgba(91, 192, 190, 0.5) !important;
    }
    div[data-baseweb="select"] *, div[data-baseweb="input"] input { color: white !important; background: transparent !important; }

    /* Style for radio buttons */
    div[role="radiogroup"] > label { color: #e0e0e0 !important; }
</style>
""", unsafe_allow_html=True)

st.title("Institutional Quant AI: Dual-Engine Architecture")

# ---- 2. QUANT PIPELINE ----
class QuantPipeline:
    def __init__(self, symbol, start_date, end_date, time_step, model_architecture):
        self.symbol = symbol
        self.start_date = start_date
        self.end_date = end_date
        self.time_step = time_step
        self.model_architecture = model_architecture
        self.scaler = RobustScaler()

    def fetch_data(self):
        try:
            df = yf.download(self.symbol, start=self.start_date, end=self.end_date, progress=False)
            if len(df) == 0: raise ValueError("No data found.")

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            req_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            if not all(col in df.columns for col in req_cols):
                 if 'Adj Close' in df.columns:
                     df['Close'] = df['Adj Close']

            return df
        except Exception as e:
            st.error(f"Data Feed Error: {e}")
            st.stop()

    def engineer_features(self, df):
        df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
        df['Target'] = df['Log_Returns'].shift(-1)

        log_hl = np.log(df['High'] / df['Low'])**2
        log_co = np.log(df['Close'] / df['Open'])**2
        df['Garman_Klass'] = 0.5 * log_hl - (2 * np.log(2) - 1) * log_co

        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        df['RSI'] = 100 - (100 / (1 + rs))

        high_low = df['High'] - df['Low']
        high_low = high_low.replace(0, 0.0001)
        ad_val = (((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / high_low) * df['Volume']
        df['CMF'] = ad_val.rolling(20).sum() / df['Volume'].rolling(20).sum()
        df['CMF'] = df['CMF'].fillna(0).replace([np.inf, -np.inf], 0)

        ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
        ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
        df['MACD'] = ema_12 - ema_26
        df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
        df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

        sma_20 = df['Close'].rolling(window=20).mean()
        std_20 = df['Close'].rolling(window=20).std()
        bb_high = sma_20 + (std_20 * 2)
        bb_low = sma_20 - (std_20 * 2)
        df['BB_Position'] = (df['Close'] - bb_low) / (bb_high - bb_low).replace(0, 0.0001)

        df['TR'] = np.maximum(df['High'] - df['Low'],
                              np.maximum(abs(df['High'] - df['Close'].shift(1)),
                                         abs(df['Low'] - df['Close'].shift(1))))
        df['ATR'] = df['TR'].rolling(14).mean() / df['Close']

        df['MOM_5'] = df['Close'].pct_change(5)
        df['MOM_20'] = df['Close'].pct_change(20)

        t = df.index
        df['Day_Sin'] = np.sin(2 * np.pi * t.dayofweek / 7)
        df['Day_Cos'] = np.cos(2 * np.pi * t.dayofweek / 7)

        df.dropna(inplace=True)
        return df

    def prepare_tensors(self, df, split_ratio=0.8):
        feature_cols = ['Log_Returns', 'Garman_Klass', 'RSI', 'CMF', 'MACD_Hist',
                        'BB_Position', 'ATR', 'MOM_5', 'MOM_20', 'Day_Sin', 'Day_Cos']

        split_idx = int(len(df) * split_ratio)
        train_df = df.iloc[:split_idx]
        test_df = df.iloc[split_idx:]

        decay_rate = 0.001
        dates = np.arange(len(train_df))
        weights = np.exp(decay_rate * dates)
        weights = weights / weights.mean()

        X_train_scaled = self.scaler.fit_transform(train_df[feature_cols])
        X_test_scaled = self.scaler.transform(test_df[feature_cols])

        y_train = train_df['Target'].values
        y_test = test_df['Target'].values

        def create_window(X, y, time_step, w=None):
            Xs, ys, ws = [], [], []
            for i in range(len(X) - time_step):
                Xs.append(X[i:(i + time_step)])
                ys.append(y[i + time_step])
                if w is not None: ws.append(w[i + time_step])
            if w is not None:
                return np.array(Xs), np.array(ys), np.array(ws)
            return np.array(Xs), np.array(ys)

        X_train_3d, y_train_3d, w_train = create_window(X_train_scaled, y_train, self.time_step, weights)
        X_test_3d, y_test_3d = create_window(X_test_scaled, y_test, self.time_step)

        return X_train_3d, y_train_3d, w_train, X_test_3d, y_test_3d, test_df.iloc[self.time_step:]

    def build_model(self, input_shape):
        tf.keras.backend.clear_session()
        gc.collect()

        inputs = Input(shape=input_shape)
        x = GaussianNoise(0.01)(inputs)
        x = BatchNormalization()(x)

        # ---- DYNAMIC ARCHITECTURE SWITCHER ----
        if self.model_architecture == "TCN + GRU (Sniper Alpha)":
            # 1. State-of-the-Art TCN Block
            conv1 = Conv1D(filters=64, kernel_size=3, dilation_rate=1, padding='causal', activation='gelu')(x)
            conv2 = Conv1D(filters=64, kernel_size=3, dilation_rate=2, padding='causal', activation='gelu')(conv1)
            conv3 = Conv1D(filters=64, kernel_size=3, dilation_rate=4, padding='causal', activation='gelu')(conv2)
            x = Add()([conv1, conv3])
            x = LayerNormalization()(x)
            # 2. Bi-GRU
            x = Bidirectional(GRU(128, return_sequences=False, kernel_regularizer=l2(1e-4)))(x)
            x = Dropout(0.4)(x)

        elif self.model_architecture == "Optimized LSTM (Deep Memory)":
            # 1. Stacked Bidirectional LSTM Block (Optimized for heavy noise reduction)
            x = Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4)))(x)
            x = LayerNormalization()(x)
            x = Dropout(0.4)(x)

            # 2. Bottleneck LSTM Layer
            x = Bidirectional(LSTM(64, return_sequences=False, kernel_regularizer=l2(1e-4)))(x)
            x = Dropout(0.4)(x)

        # ---- SHARED DENSE HEAD ----
        x = Dense(64, activation='gelu', kernel_regularizer=l2(1e-4))(x)
        x = Dropout(0.3)(x)

        outputs = Dense(1, activation='linear', dtype='float32')(x)

        model = Model(inputs=inputs, outputs=outputs)
        optimizer = Adam(learning_rate=0.0005)
        model.compile(optimizer=optimizer, loss='huber')

        return model


# ---- 3. UI & PRESET CONFIGURATIONS ----
IDEAL_CONFIGS = {
    "AAPL": {"start": "2018-01-01", "long_only": True, "cost_bps": 10, "target_vol": 30, "threshold": 0.20},
    "NVDA": {"start": "2018-01-01", "long_only": True, "cost_bps": 10, "target_vol": 60, "threshold": 0.30},
    "MSFT": {"start": "2018-01-01", "long_only": True, "cost_bps": 10, "target_vol": 30, "threshold": 0.20},
    "SPY": {"start": "2010-01-01", "long_only": True, "cost_bps": 5, "target_vol": 15, "threshold": 0.10},
    "BTC-USD": {"start": "2019-01-01", "long_only": False, "cost_bps": 20, "target_vol": 80, "threshold": 0.40},
    "ETH-USD": {"start": "2019-01-01", "long_only": False, "cost_bps": 20, "target_vol": 90, "threshold": 0.40},
}

with st.sidebar:
    st.header("1. Core System Config")
    ticker = st.selectbox("Asset Class", list(IDEAL_CONFIGS.keys()))
    cfg = IDEAL_CONFIGS[ticker]

    # ---- NEW NEURAL NETWORK SELECTOR ----
    st.markdown("---")
    st.header("2. Neural Engine")
    model_arch = st.radio(
        "Select Core Architecture:",
        ("TCN + GRU (Sniper Alpha)", "Optimized LSTM (Deep Memory)"),
        help="Compare state-of-the-art Causal Convolutions against traditional Deep Recurrent memory."
    )

    st.markdown("---")
    st.header("3. Risk Management")
    start = st.date_input("Start Date", value=pd.to_datetime(cfg["start"]))
    end = st.date_input("End Date", value=pd.to_datetime("today"))

    long_only = st.checkbox("Long Only Mode (No Shorts)", value=cfg["long_only"])
    ensemble_size = st.slider("Ensemble Size", 1, 10, 5)
    cost_bps = st.slider("Transaction Cost (bps)", 0, 50, cfg["cost_bps"])
    target_vol = st.slider("Target Volatility (%)", 10, 100, cfg["target_vol"])

    conviction_thresh = st.slider("Conviction Deadband", 0.0, 1.0, cfg["threshold"],
                                  help="If AI conviction is below this, it holds cash. Saves massive fees.")

if st.button(f"Launch {model_arch} Strategy"):
    # Pass the selected architecture into the Pipeline
    pipeline = QuantPipeline(ticker, start, end, 60, model_arch)

    with st.status(f"Initializing {model_arch} Engine...", expanded=True) as status:

        st.write("Fetching institutional data feed...")
        raw_df = pipeline.fetch_data()

        st.write("Calculating Advanced Alpha Features...")
        processed_df = pipeline.engineer_features(raw_df)

        st.write("Applying Exponential Decay Weights...")
        X_train, y_train, w_train, X_test, y_test, test_context = pipeline.prepare_tensors(processed_df)

        st.write("Constructing high-throughput GPU data pipelines...")
        BATCH_SIZE = 128

        train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train, w_train))
        train_dataset = train_dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
        test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        st.write(f"Training Sniper Ensemble of {ensemble_size} {model_arch.split()[0]} Models...")
        ensemble_preds = []
        progress_bar = st.progress(0)

        for i in range(ensemble_size):
            tf.random.set_seed(42 + i)
            st.write(f"Training Neural Network {i+1}/{ensemble_size} on GPU Tensor Cores")

            model = pipeline.build_model((X_train.shape[1], X_train.shape[2]))

            early_stop = EarlyStopping(monitor='val_loss', patience=18, restore_best_weights=True)
            reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=6, min_lr=1e-6)

            model.fit(
                train_dataset,
                validation_data=test_dataset,
                epochs=60,
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )

            preds = model.predict(test_dataset, verbose=0).flatten()
            ensemble_preds.append(preds)
            progress_bar.progress((i + 1) / ensemble_size)

        st.write("Aggregating predictions & Applying Conviction Deadband...")
        avg_preds = np.mean(ensemble_preds, axis=0)

        status.update(label="Optimization Complete", state="complete", expanded=False)

    # ---- 4. BACKTEST ENGINE ----
    backtest_df = test_context.copy()
    backtest_df['Predicted_Return'] = avg_preds
    backtest_df['Actual_Return'] = y_test

    backtest_df['SMA_50'] = backtest_df['Close'].rolling(window=50).mean()
    backtest_df['Trend_Bullish'] = backtest_df['Close'] > backtest_df['SMA_50']

    current_vol = np.sqrt(backtest_df['Garman_Klass']) * np.sqrt(252)
    safe_vol = (current_vol * 100).replace(0, 0.01)
    backtest_df['Vol_Scalar'] = target_vol / safe_vol
    backtest_df['Vol_Scalar'] = backtest_df['Vol_Scalar'].clip(upper=2.0)

    boost_factor = np.where(backtest_df['Trend_Bullish'], 2.0, 1.0)

    backtest_df['Conviction'] = tf.math.sigmoid(backtest_df['Predicted_Return'] * 500 * boost_factor).numpy() - 0.5
    backtest_df['Conviction'] = backtest_df['Conviction'] * 2

    backtest_df['Conviction'] = np.where(backtest_df['Conviction'].abs() >= conviction_thresh, backtest_df['Conviction'], 0.0)

    raw_position = backtest_df['Conviction'] * backtest_df['Vol_Scalar']

    if long_only:
        backtest_df['Position'] = raw_position.clip(lower=0)
    else:
        backtest_df['Position'] = raw_position

    backtest_df['Position'] = backtest_df['Position'].fillna(0)

    backtest_df['Turnover'] = backtest_df['Position'].diff().abs().fillna(0)
    backtest_df['Cost'] = backtest_df['Turnover'] * (cost_bps / 10000)

    backtest_df['Gross_Return'] = backtest_df['Position'] * backtest_df['Actual_Return']
    backtest_df['Net_Return'] = backtest_df['Gross_Return'] - backtest_df['Cost']

    backtest_df['Cum_Market'] = (1 + backtest_df['Actual_Return']).cumprod()
    backtest_df['Cum_Strategy'] = (1 + backtest_df['Net_Return']).cumprod()

    total_ret = backtest_df['Cum_Strategy'].iloc[-1] - 1
    sharpe = np.sqrt(252) * (backtest_df['Net_Return'].mean() / backtest_df['Net_Return'].std())
    peak = backtest_df['Cum_Strategy'].cummax()
    max_dd = ((backtest_df['Cum_Strategy'] - peak) / peak).min()

    wins = backtest_df[backtest_df['Net_Return'] > 0]['Net_Return']
    backtest_df['Realized_Correctness'] = np.sign(backtest_df['Predicted_Return']) == np.sign(backtest_df['Net_Return'])

    active_trading_days = backtest_df[backtest_df['Position'].abs() > 0]

    win_rate = len(wins) / len(active_trading_days) if len(active_trading_days) > 0 else 0
    accuracy = active_trading_days['Realized_Correctness'].mean() * 100 if len(active_trading_days) > 0 else 0

    st.divider()

    st.container()
    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Net Profit", f"{total_ret*100:.2f}%")
    c2.metric("Sharpe Ratio", f"{sharpe:.2f}")
    c3.metric("Win Rate", f"{win_rate*100:.1f}%")
    c4.metric("Max Drawdown", f"{max_dd*100:.2f}%")
    c5.metric("Realized Accuracy", f"{accuracy:.1f}%")

    st.markdown("<br>", unsafe_allow_html=True)
    st.subheader(f"Performance (Net of Fees) - {model_arch}")
    st.line_chart(backtest_df[['Cum_Strategy', 'Cum_Market']])

    with st.expander("See Trade Log"):
        st.dataframe(backtest_df[['Predicted_Return', 'Conviction', 'Position', 'Cost', 'Net_Return']].tail(100))
        csv = backtest_df.to_csv().encode('utf-8')
        st.download_button("Download Data", csv, f"backtest_data_{model_arch.split()[0]}.csv", "text/csv")

Overwriting app.py


In [ ]:
import subprocess
import time
import os
from google.colab import output
from google.colab.output import eval_js

print("Cleaning up old background processes...")
os.system("pkill -f streamlit")
time.sleep(1)

print("Starting Streamlit server...")
# 1. We removed the DEVNULL silencers so we can see if it crashes
# 2. We added back the 0.0.0.0 binding and CORS bypass
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.headless", "true",
    "--server.address", "0.0.0.0",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# Give the server 5 full seconds to initialize before the proxy connects
time.sleep(5)

print("Opening native Colab proxy...")
proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"Click here for full-screen dashboard: {proxy_url}")

print("Loading embedded dashboard...")
output.serve_kernel_port_as_iframe(8501, height=720)

Cleaning up old background processes...
Starting Streamlit server...


FileNotFoundError: [Errno 2] No such file or directory: 'streamlit'